In [1]:
import os
import pandas as pd
import numpy as np
import boto3
from tqdm import tqdm
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
# Suppress PerformanceWarning
warnings.filterwarnings('ignore')
import sklearn.metrics as skm

try:
    import catboost as cb
except:
    ! pip install catboost

### Functions

In [2]:
# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # download file
    boto3.client('s3').download_file(str_project, str_bucket_path, str_local_path)

In [3]:
# upload to s3
def upload_to_s3(str_local_path, str_bucket_key, str_bucket_name):
    # upload file
    boto3.client('s3').upload_file(str_project, str_bucket_path, str_local_path)

In [4]:
# get tier
def get_tier(fltDebtorScore):
    if fltDebtorScore <= 0.04:
        return 'A1'
    elif fltDebtorScore <= 0.07:
        return 'A'
    elif fltDebtorScore <= 0.14:
        return 'B'
    elif fltDebtorScore <= 0.17:
        return 'C'
    elif fltDebtorScore <= 0.2015:
        return 'D'
    else:
        return 'Decline'

### Constants

In [5]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')
str_dirname_output = './output'
str_variant = 'noPTImodel7'

# indicators
dict_model_column = {
    'DQ1_1': 'Early_Pay_Delinquency_1_30_Flag',
    'DQ1_2': 'Early_Pay_Delinquency_1_60_Flag',
    'DQ1_3': 'Early_Pay_Delinquency_1_90_Flag',
    'DQ1_4': 'Early_Pay_Delinquency_1_120_Flag',
    'DQ1_5': 'Early_Pay_Delinquency_1_150_Flag',
    'DQ1_6': 'Early_Pay_Delinquency_1_180_Flag',
    'DQ1_7': 'Early_Pay_Delinquency_1_210_Flag',
    'DQ1_8': 'Early_Pay_Delinquency_1_240_Flag',
    'DQ1_9': 'Early_Pay_Delinquency_1_270_Flag',
    'DQ1_10': 'Early_Pay_Delinquency_1_300_Flag',
    'DQ1_11': 'Early_Pay_Delinquency_1_330_Flag',
    'DQ1_12': 'Early_Pay_Delinquency_1_360_Flag',
    'DQ1_13': 'Early_Pay_Delinquency_1_390_Flag',
    'DQ1_14': 'Early_Pay_Delinquency_1_420_Flag',
    'DQ1_15': 'Early_Pay_Delinquency_1_450_Flag',
    'DQ1_16': 'Early_Pay_Delinquency_1_480_Flag',
    'DQ1_17': 'Early_Pay_Delinquency_1_510_Flag',
    'DQ1_18': 'Early_Pay_Delinquency_1_540_Flag',
    'DQ1_19': 'Early_Pay_Delinquency_1_570_Flag',
    'DQ1_20': 'Early_Pay_Delinquency_1_600_Flag',
    'DQ1_21': 'Early_Pay_Delinquency_1_630_Flag',
    'DQ1_22': 'Early_Pay_Delinquency_1_660_Flag',
    'DQ1_23': 'Early_Pay_Delinquency_1_690_Flag',
    'DQ1_24': 'Early_Pay_Delinquency_1_720_Flag',
    'DQ15_1': 'Early_Pay_Delinquency_15_30_Flag',
    'DQ15_2': 'Early_Pay_Delinquency_15_60_Flag',
    'DQ15_3': 'Early_Pay_Delinquency_15_90_Flag',
    'DQ15_4': 'Early_Pay_Delinquency_15_120_Flag',
    'DQ15_5': 'Early_Pay_Delinquency_15_150_Flag',
    'DQ15_6': 'Early_Pay_Delinquency_15_180_Flag',
    'DQ15_7': 'Early_Pay_Delinquency_15_210_Flag',
    'DQ15_8': 'Early_Pay_Delinquency_15_240_Flag',
    'DQ15_9': 'Early_Pay_Delinquency_15_270_Flag',
    'DQ15_10': 'Early_Pay_Delinquency_15_300_Flag',
    'DQ15_11': 'Early_Pay_Delinquency_15_330_Flag',
    'DQ15_12': 'Early_Pay_Delinquency_15_360_Flag',
    'DQ15_13': 'Early_Pay_Delinquency_15_390_Flag',
    'DQ15_14': 'Early_Pay_Delinquency_15_420_Flag',
    'DQ15_15': 'Early_Pay_Delinquency_15_450_Flag',
    'DQ15_16': 'Early_Pay_Delinquency_15_480_Flag',
    'DQ15_17': 'Early_Pay_Delinquency_15_510_Flag',
    'DQ15_18': 'Early_Pay_Delinquency_15_540_Flag',
    'DQ15_19': 'Early_Pay_Delinquency_15_570_Flag',
    'DQ15_20': 'Early_Pay_Delinquency_15_600_Flag',
    'DQ15_21': 'Early_Pay_Delinquency_15_630_Flag',
    'DQ15_22': 'Early_Pay_Delinquency_15_660_Flag',
    'DQ15_23': 'Early_Pay_Delinquency_15_690_Flag',
    'DQ15_24': 'Early_Pay_Delinquency_15_720_Flag',
    'DQ30_1': 'Early_Pay_Delinquency_30_30_Flag',
    'DQ30_2': 'Early_Pay_Delinquency_30_60_Flag',
    'DQ30_3': 'Early_Pay_Delinquency_30_90_Flag',
    'DQ30_4': 'Early_Pay_Delinquency_30_120_Flag',
    'DQ30_5': 'Early_Pay_Delinquency_30_150_Flag',
    'DQ30_6': 'Early_Pay_Delinquency_30_180_Flag',
    'DQ30_7': 'Early_Pay_Delinquency_30_210_Flag',
    'DQ30_8': 'Early_Pay_Delinquency_30_240_Flag',
    'DQ30_9': 'Early_Pay_Delinquency_30_270_Flag',
    'DQ30_10': 'Early_Pay_Delinquency_30_300_Flag',
    'DQ30_11': 'Early_Pay_Delinquency_30_330_Flag',
    'DQ30_12': 'Early_Pay_Delinquency_30_360_Flag',
    'DQ30_13': 'Early_Pay_Delinquency_30_390_Flag',
    'DQ30_14': 'Early_Pay_Delinquency_30_420_Flag',
    'DQ30_15': 'Early_Pay_Delinquency_30_450_Flag',
    'DQ30_16': 'Early_Pay_Delinquency_30_480_Flag',
    'DQ30_17': 'Early_Pay_Delinquency_30_510_Flag',
    'DQ30_18': 'Early_Pay_Delinquency_30_540_Flag',
    'DQ30_19': 'Early_Pay_Delinquency_30_570_Flag',
    'DQ30_20': 'Early_Pay_Delinquency_30_600_Flag',
    'DQ30_21': 'Early_Pay_Delinquency_30_630_Flag',
    'DQ30_22': 'Early_Pay_Delinquency_30_660_Flag',
    'DQ30_23': 'Early_Pay_Delinquency_30_690_Flag',
    'DQ30_24': 'Early_Pay_Delinquency_30_720_Flag',
    'DQ60_3': 'Early_Pay_Delinquency_60_90_Flag',
    'DQ60_4': 'Early_Pay_Delinquency_60_120_Flag',
    'DQ60_5': 'Early_Pay_Delinquency_60_150_Flag',
    'DQ60_6': 'Early_Pay_Delinquency_60_180_Flag',
    'DQ60_7': 'Early_Pay_Delinquency_60_210_Flag',
    'DQ60_8': 'Early_Pay_Delinquency_60_240_Flag',
    'DQ60_9': 'Early_Pay_Delinquency_60_270_Flag',
    'DQ60_10': 'Early_Pay_Delinquency_60_300_Flag',
    'DQ60_11': 'Early_Pay_Delinquency_60_330_Flag',
    'DQ60_12': 'Early_Pay_Delinquency_60_360_Flag',
    'DQ60_13': 'Early_Pay_Delinquency_60_390_Flag',
    'DQ60_14': 'Early_Pay_Delinquency_60_420_Flag',
    'DQ60_15': 'Early_Pay_Delinquency_60_450_Flag',
    'DQ60_16': 'Early_Pay_Delinquency_60_480_Flag',
    'DQ60_17': 'Early_Pay_Delinquency_60_510_Flag',
    'DQ60_18': 'Early_Pay_Delinquency_60_540_Flag',
    'DQ60_19': 'Early_Pay_Delinquency_60_570_Flag',
    'DQ60_20': 'Early_Pay_Delinquency_60_600_Flag',
    'DQ60_21': 'Early_Pay_Delinquency_60_630_Flag',
    'DQ60_22': 'Early_Pay_Delinquency_60_660_Flag',
    'DQ60_23': 'Early_Pay_Delinquency_60_690_Flag',
    'DQ60_24': 'Early_Pay_Delinquency_60_720_Flag',
    'DQ90_4': 'Early_Pay_Delinquency_90_120_Flag',
    'DQ90_5': 'Early_Pay_Delinquency_90_150_Flag',
    'DQ90_6': 'Early_Pay_Delinquency_90_180_Flag',
    'DQ90_7': 'Early_Pay_Delinquency_90_210_Flag',
    'DQ90_8': 'Early_Pay_Delinquency_90_240_Flag',
    'DQ90_9': 'Early_Pay_Delinquency_90_270_Flag',
    'DQ90_10': 'Early_Pay_Delinquency_90_300_Flag',
    'DQ90_11': 'Early_Pay_Delinquency_90_330_Flag',
    'DQ90_12': 'Early_Pay_Delinquency_90_360_Flag',
    'DQ90_13': 'Early_Pay_Delinquency_90_390_Flag',
    'DQ90_14': 'Early_Pay_Delinquency_90_420_Flag',
    'DQ90_15': 'Early_Pay_Delinquency_90_450_Flag',
    'DQ90_16': 'Early_Pay_Delinquency_90_480_Flag',
    'DQ90_17': 'Early_Pay_Delinquency_90_510_Flag',
    'DQ90_18': 'Early_Pay_Delinquency_90_540_Flag',
    'DQ90_19': 'Early_Pay_Delinquency_90_570_Flag',
    'DQ90_20': 'Early_Pay_Delinquency_90_600_Flag',
    'DQ90_21': 'Early_Pay_Delinquency_90_630_Flag',
    'DQ90_22': 'Early_Pay_Delinquency_90_660_Flag',
    'DQ90_23': 'Early_Pay_Delinquency_90_690_Flag',
    'DQ90_24': 'Early_Pay_Delinquency_90_720_Flag',
}

# months (production data goes from 2021-09-27 to 2023-11-27)
list_str_year_month = [
    '2021-10',
    '2021-11',
    '2021-12',
    '2022-01',
    '2022-02',
#     '2022-03',
#     '2022-04',
#     '2022-05',
#     '2022-06',
#     '2022-07',
#     '2022-08',
#     '2022-09',
#     '2022-10',
#     '2022-11',
#     '2022-12',
#     '2023-01',
#     '2023-02',
#     '2023-03',
#     '2023-04',
#     '2023-05',
#     '2023-06',
#     '2023-07',
#     '2023-08',
#     '2023-09',
#     '2023-10',
]
# note: production targets were pulled in 2024-02
# maximum days total of the targets is 720
# thus, to ensure all accounts have been 720 days (24 months) mature, I have subsetted to the latest date being 2022-02
# we can edit this as needed

Project: 20231010-gen-xii
Task: 09_early_indicators


### Output directory

In [6]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Variant directory

In [7]:
try:
    os.mkdir(f'{str_dirname_output}/{str_variant}')
except:
    pass

### Get mean of target in training data

In [8]:
# load raw pd training data 
str_filename = 'df_train_raw.gzip'
str_uri = f's3://{str_project}/02_pricing_pd/01_data_prep/03_train_valid_test_split/{str_filename}'
df_tmp = pd.read_parquet(
    str_uri, 
)
df_tmp['uniqueid'] = df_tmp['uniqueid'].astype(int)
df_tmp.drop_duplicates(subset='uniqueid', keep='last', inplace=True)
# show
df_tmp

,dtmstampcreation__base,dtmapproved__base,dtmdeclined__base,observationdate__base,analyticsmatchkey__base,decision_dte__base,booked__base,acct_typ_cde__base,open_dte__base,curr_bal_amt__base,...,fltdowncash__app,fltapproveddowntotal__app,payment__app,dti__app,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2000.0,2000.0,398.83,0.229354,0.052130,1,1.106635,auto,0,2012-06-18 09:31:54.393
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,420.66,0.470570,0.153369,1,1.254334,auto,1,2012-03-06 16:36:21.623
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,440.04,0.405641,0.050115,0,1.249982,auto,0,2009-10-15 16:06:40.767
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,483.21,0.323628,0.137320,1,1.348074,auto,0,2012-11-28 17:21:08.587
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1500.0,1500.0,530.00,0.353894,0.128363,0,0.936749,auto,0,2010-02-19 17:19:40.037
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74583,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,420.00,0.337275,0.159882,0,1.242583,auto,1,2003-07-08 14:32:29.097
74584,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1500.0,1500.0,699.86,0.430698,0.150794,1,1.082799,suv,0,2007-09-18 17:16:17.027
74586,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,428.56,0.344195,0.081817,1,1.128015,auto,1,2012-04-26 16:31:49.550
74587,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1000.0,1000.0,324.51,0.493262,0.107683,0,1.143420,auto,0,2012-03-06 16:36:21.623


In [9]:
# preprocess
list_str_filename = [
    'preprocessing.py',
    'cls_model_preprocessing.pkl',
]
for str_filename in tqdm(list_str_filename):
    # download
    str_bucket_path = f'01_ad/02_model/{str_variant}/00_preprocessing/01_create_preprocessor/{str_filename}'
    str_local_path = f'./{str_filename}'
    download_from_s3(
        str_local_path=str_local_path, 
        str_bucket_path=str_bucket_path, 
        str_project=str_project,
    )
# import
cls_model_preprocessing = pickle.load(open(str_local_path, 'rb'))
# rm
os.remove(str_local_path)

# preprocess
df_tmp = cls_model_preprocessing.transform(df_tmp)

# rm
os.remove('./preprocessing.py')

# show
df_tmp

100%|██████████| 2/2 [00:00<00:00,  5.66it/s]


NaN Replacer: 1.4122 sec.


100%|██████████| 3/3 [00:00<00:00, 49.74it/s]

Set strings: 0.062305 sec.


Boolean Replacer: 1.6078 sec.


...


100%|██████████| 2483/2483 [00:02<00:00, 1142.30it/s]


Replace inf and -inf with NaN: 2.5908 sec.
Imputer: 1.2465 sec.
Map term: 0.030312 sec.
Map PTI: 0.033128 sec.


100%|██████████| 9/9 [00:00<00:00, 1353.49it/s]

Round values: 0.010342 sec.
Preprocessing Model: 13.957 sec.


,dtmstampcreation__base,dtmapproved__base,dtmdeclined__base,observationdate__base,analyticsmatchkey__base,decision_dte__base,booked__base,acct_typ_cde__base,open_dte__base,curr_bal_amt__base,...,bitgap__app,dealerstampcreation__app,year,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-payment_to_income,ENG-loan_to_value,ENG-vehicle_age,ENG-dealership_age
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,2012-06-18 09:31:54.393,2013,1.256262,10,4,0.09,1.370370,4.0,1.284932
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1,2012-03-06 16:36:21.623,2013,1.256262,10,4,0.15,1.586207,2.0,1.569863
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,2009-10-15 16:06:40.767,2013,1.256262,10,4,0.03,1.342857,1.0,3.961644
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,2012-11-28 17:21:08.587,2013,1.256262,10,4,0.12,1.588235,2.0,0.838356
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,2010-02-19 17:19:40.037,2013,1.256262,10,4,0.12,1.037037,0.0,3.613699
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74583,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1,2003-07-08 14:32:29.097,2016,1.219360,3,1,0.15,1.482759,1.0,12.701370
74584,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,2007-09-18 17:16:17.027,2016,1.219360,3,1,0.15,1.222222,3.0,8.498630
74586,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1,2012-04-26 16:31:49.550,2016,1.219360,3,1,0.09,1.343750,1.0,3.890411
74587,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,2012-03-06 16:36:21.623,2016,1.219360,3,1,0.09,1.240000,5.0,4.030137


In [10]:
# get gen 12 PD model
str_filename = 'final_model.pkl'
str_bucket_path = f'02_pricing_pd/02_model/{str_variant}/03_final_model/{str_filename}'
str_local_path = f'{str_dirname_output}/{str_filename}'
download_from_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)
cls_model_inference = pickle.load(open(str_local_path, 'rb'))['model_inference']
os.remove(str_local_path)
# predict
list_cols_model = list(cls_model_inference.feature_names_)
df_tmp['yhat_pd'] = cls_model_inference.predict_proba(df_tmp[list_cols_model])[:, 1]
# show
df_tmp

,dtmstampcreation__base,dtmapproved__base,dtmdeclined__base,observationdate__base,analyticsmatchkey__base,decision_dte__base,booked__base,acct_typ_cde__base,open_dte__base,curr_bal_amt__base,...,dealerstampcreation__app,year,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-payment_to_income,ENG-loan_to_value,ENG-vehicle_age,ENG-dealership_age,yhat_pd
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2012-06-18 09:31:54.393,2013,1.256262,10,4,0.09,1.370370,4.0,1.284932,0.513062
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2012-03-06 16:36:21.623,2013,1.256262,10,4,0.15,1.586207,2.0,1.569863,0.310533
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2009-10-15 16:06:40.767,2013,1.256262,10,4,0.03,1.342857,1.0,3.961644,0.706639
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2012-11-28 17:21:08.587,2013,1.256262,10,4,0.12,1.588235,2.0,0.838356,0.448451
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2010-02-19 17:19:40.037,2013,1.256262,10,4,0.12,1.037037,0.0,3.613699,0.241002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74583,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2003-07-08 14:32:29.097,2016,1.219360,3,1,0.15,1.482759,1.0,12.701370,0.689931
74584,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2007-09-18 17:16:17.027,2016,1.219360,3,1,0.15,1.222222,3.0,8.498630,0.172488
74586,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2012-04-26 16:31:49.550,2016,1.219360,3,1,0.09,1.343750,1.0,3.890411,0.240983
74587,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2012-03-06 16:36:21.623,2016,1.219360,3,1,0.09,1.240000,5.0,4.030137,0.310571


In [11]:
# get gen 12 LGD model
str_filename = 'final_model.pkl'
str_bucket_path = f'03_pricing_lgd/02_model/{str_variant}/03_final_model/{str_filename}'
str_local_path = f'{str_dirname_output}/{str_filename}'
download_from_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)
cls_model_inference = pickle.load(open(str_local_path, 'rb'))['model_inference']
os.remove(str_local_path)
# predict
list_cols_model = list(cls_model_inference.feature_names_)
df_tmp['yhat_lgd'] = cls_model_inference.predict(df_tmp[list_cols_model])
# show
df_tmp

,dtmstampcreation__base,dtmapproved__base,dtmdeclined__base,observationdate__base,analyticsmatchkey__base,decision_dte__base,booked__base,acct_typ_cde__base,open_dte__base,curr_bal_amt__base,...,year,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-payment_to_income,ENG-loan_to_value,ENG-vehicle_age,ENG-dealership_age,yhat_pd,yhat_lgd
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2013,1.256262,10,4,0.09,1.370370,4.0,1.284932,0.513062,0.419317
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2013,1.256262,10,4,0.15,1.586207,2.0,1.569863,0.310533,0.346931
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2013,1.256262,10,4,0.03,1.342857,1.0,3.961644,0.706639,0.471772
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2013,1.256262,10,4,0.12,1.588235,2.0,0.838356,0.448451,0.360220
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2013,1.256262,10,4,0.12,1.037037,0.0,3.613699,0.241002,0.434562
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74583,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2016,1.219360,3,1,0.15,1.482759,1.0,12.701370,0.689931,0.461088
74584,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2016,1.219360,3,1,0.15,1.222222,3.0,8.498630,0.172488,0.306691
74586,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2016,1.219360,3,1,0.09,1.343750,1.0,3.890411,0.240983,0.263744
74587,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2016,1.219360,3,1,0.09,1.240000,5.0,4.030137,0.310571,0.312804


In [12]:
# get ecnl
df_tmp_grouped = df_tmp.groupby(by='bigaccountid__app', as_index=False).agg({
    'yhat_pd': 'mean',
    'yhat_lgd': 'mean',
})
df_tmp_grouped['ecnl'] = df_tmp_grouped['yhat_pd'] * df_tmp_grouped['yhat_lgd']

# Mike's adjustment
df_tmp_grouped['ecnl'] = (1.95553 * df_tmp_grouped['ecnl']) - 0.03281

# get tier
df_tmp_grouped['tier'] = df_tmp_grouped['ecnl'].apply(get_tier)
# subset
list_cols = [
    'bigaccountid__app',
    'tier',
]
df_tmp_grouped = df_tmp_grouped[list_cols].copy()
# show
df_tmp_grouped

,bigaccountid__app,tier
0,1337511.0,Decline
1,1337528.0,D
2,1337539.0,Decline
3,1337542.0,Decline
4,1337547.0,D
...,...,...
59036,2493925.0,Decline
59037,2493958.0,B
59038,2493969.0,B
59039,2493994.0,C


In [13]:
# join back to df_tmp
df_tmp = pd.merge(
    left=df_tmp,
    right=df_tmp_grouped,
    on='bigaccountid__app',
    how='inner',
)
df_tmp

,dtmstampcreation__base,dtmapproved__base,dtmdeclined__base,observationdate__base,analyticsmatchkey__base,decision_dte__base,booked__base,acct_typ_cde__base,open_dte__base,curr_bal_amt__base,...,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-payment_to_income,ENG-loan_to_value,ENG-vehicle_age,ENG-dealership_age,yhat_pd,yhat_lgd,tier
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.256262,10,4,0.09,1.370370,4.0,1.284932,0.513062,0.419317,Decline
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.256262,10,4,0.15,1.586207,2.0,1.569863,0.310533,0.346931,D
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.256262,10,4,0.03,1.342857,1.0,3.961644,0.706639,0.471772,Decline
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.256262,10,4,0.12,1.588235,2.0,0.838356,0.448451,0.360220,Decline
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.256262,10,4,0.12,1.037037,0.0,3.613699,0.241002,0.434562,D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59036,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.219360,3,1,0.15,1.482759,1.0,12.701370,0.689931,0.461088,Decline
59037,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.219360,3,1,0.15,1.222222,3.0,8.498630,0.172488,0.306691,B
59038,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.219360,3,1,0.09,1.343750,1.0,3.890411,0.240983,0.263744,B
59039,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.219360,3,1,0.09,1.240000,5.0,4.030137,0.310571,0.312804,C


In [14]:
# subset to tier
df_tmp = df_tmp[df_tmp['tier'] == 'A1'].copy()
# subse to bk type
df_tmp = df_tmp[df_tmp['intopenbktype__app'] == 'nan'].copy()
# show
df_tmp

,dtmstampcreation__base,dtmapproved__base,dtmdeclined__base,observationdate__base,analyticsmatchkey__base,decision_dte__base,booked__base,acct_typ_cde__base,open_dte__base,curr_bal_amt__base,...,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-payment_to_income,ENG-loan_to_value,ENG-vehicle_age,ENG-dealership_age,yhat_pd,yhat_lgd,tier
47,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.256262,10,4,0.12,1.209677,1.0,6.279452,0.129057,0.247420,A1
53,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.256262,10,4,0.15,1.171429,4.0,0.435616,0.044347,0.265993,A1
124,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.256262,10,4,0.06,1.293103,0.0,7.068493,0.022384,0.218611,A1
229,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.256262,10,4,0.15,0.933333,4.0,2.378082,0.074813,0.300604,A1
295,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.256262,10,4,0.09,1.343750,1.0,0.389041,0.128835,0.269406,A1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58699,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.219360,3,1,0.09,1.243243,3.0,4.654795,0.096104,0.364405,A1
58752,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.219360,3,1,0.15,1.116279,1.0,4.800000,0.173298,0.194351,A1
58781,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.219360,3,1,0.09,1.309524,5.0,5.463014,0.102708,0.314343,A1
58832,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.219360,3,1,0.06,1.666667,6.0,9.506849,0.066530,0.254901,A1


In [15]:
# get targets
str_filename = 'df_monitoring_targets.csv'
str_uri = f's3://{str_project}/09_early_indicators/input/{str_filename}'
df_tmp2 = pd.read_csv(str_uri)
df_tmp2['UniqueID'] = df_tmp2['UniqueID'].astype(int)
df_tmp2.drop_duplicates(subset='UniqueID', keep='last', inplace=True)

# join
df_tmp = pd.merge(
    left=df_tmp,
    right=df_tmp2,
    left_on='uniqueid',
    right_on='UniqueID',
    how='inner'
)

# get means
dict_train_target_mean = {}
for str_col in tqdm(dict_model_column.keys()):
    # get mean
    flt_mean = df_tmp[str_col].mean()
    # assign
    dict_train_target_mean[str_col] = flt_mean

# save memory
del df_tmp

# show
dict_train_target_mean

100%|██████████| 115/115 [00:00<00:00, 15696.73it/s]


{'DQ1_1': 0.09533201840894148,
 'DQ1_2': 0.1913214990138067,
 'DQ1_3': 0.27087442472057854,
 'DQ1_4': 0.3510848126232742,
 'DQ1_5': 0.4069690992767916,
 'DQ1_6': 0.44904667981591057,
 'DQ1_7': 0.4930966469428008,
 'DQ1_8': 0.5154503616042078,
 'DQ1_9': 0.5424063116370809,
 'DQ1_10': 0.5548980933596318,
 'DQ1_11': 0.5706771860618014,
 'DQ1_12': 0.5838264299802761,
 'DQ1_13': 0.5976331360946746,
 'DQ1_14': 0.6061801446416831,
 'DQ1_15': 0.6140696909927679,
 'DQ1_16': 0.6186719263642341,
 'DQ1_17': 0.62853385930309,
 'DQ1_18': 0.6416831032215647,
 'DQ1_19': 0.646285338593031,
 'DQ1_20': 0.6515450361604208,
 'DQ1_21': 0.6561472715318869,
 'DQ1_22': 0.6600920447074293,
 'DQ1_23': 0.666009204470743,
 'DQ1_24': 0.6719263642340565,
 'DQ15_1': 0.0019723865877712033,
 'DQ15_2': 0.005917159763313609,
 'DQ15_3': 0.014464168310322156,
 'DQ15_4': 0.023668639053254437,
 'DQ15_5': 0.032215647600262985,
 'DQ15_6': 0.03550295857988166,
 'DQ15_7': 0.042735042735042736,
 'DQ15_8': 0.04930966469428008,
 'D

### Load data from retro scoring

In [16]:
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/08_retro_scoring/06_create_df/{str_filename}'
df = pd.read_parquet(str_uri)

# convert to datetime
df['applicationdate__app'] = pd.to_datetime(df['applicationdate__app'])

# drop because they break preprocessing
list_cols = [
    'applicationdayofweek__app',
]
df.drop(list_cols, axis=1, inplace=True)

# get features 100% NaN and drop
ser_isnull = df.isnull().mean()
list_all_nan = list(ser_isnull[ser_isnull==1.0].index)
# logic
if str_variant == 'model3':
    list_all_nan = [col for col in list_all_nan if col != 'cvlst_s1__tu']
else:
    pass
df.drop(list_all_nan, axis=1, inplace=True)

# get month of application
df['year_month'] = df['applicationdate__app'].dt.strftime('%Y-%m')
# subset
df = df[df['year_month'].isin(list_str_year_month)].copy()
# show
df

,uniqueid__app_x,strcity__app,strname__app,strzipcode__app,bitapproved__app,bitsystemdecline__app,bitfunded__app,applicationmonth__app,applicationquarter__app,bigdealerid__app,...,fltapproveddowntotal__app,payment__app,dti__app,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app,year_month
0,5838462__7328771__20211124,Burlington,Kentucky,41005,True,nan,True,11,4,2836.0,...,500.0,484.52,0.348085,0.124236,0,0.955681,auto,0,2013-05-08 08:47:59.997,2021-11
1,5874043__7369806__20211220,Dallas,Texas,75219,True,nan,True,12,4,1469.0,...,150.0,599.41,0.413636,0.137929,0,1.118242,suv,1,2010-07-07 16:51:47.837,2021-12
2,5827507__7315370__20211113,SAINT LOUIS,Missouri,63112,True,nan,True,11,4,5473.0,...,1000.0,438.12,0.320558,0.110788,0,1.092707,auto,1,2018-05-07 13:27:18.220,2021-11
3,5855839__7349888__20211211,COLLINSVILLE,Illinois,62234,True,nan,True,12,4,5235.0,...,2000.0,386.01,0.364331,0.122209,0,1.097128,auto,1,2017-09-18 16:02:07.110,2021-12
4,5874497__7370346__20211220,CHICAGO,Illinois,60652,True,nan,True,12,4,4535.0,...,1500.0,680.00,0.160800,0.068000,0,0.957552,suv,0,2016-03-15 15:36:18.820,2021-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9417,0__0__20220222,CHANDLER,Arizona,85225,False,0.0,False,2,1,NaN,...,1000.0,510.48,0.349799,0.149798,0,1.122614,suv,0,2003-07-08 14:32:29.097,2022-02
9525,0__0__20220221,Indianapolis,Indiana,46237,False,0.0,False,2,1,NaN,...,1500.0,424.84,0.474754,0.073374,0,0.917676,suv,0,2013-07-18 09:37:34.627,2022-02
9702,0__0__20220222,Louisville,Kentucky,40272,False,0.0,False,2,1,NaN,...,0.0,529.30,0.436436,0.112239,0,0.923187,auto,1,2012-11-21 13:38:42.487,2022-02
10670,0__0__20220214,CYPRESS,Louisiana,71457,False,0.0,False,2,1,NaN,...,0.0,575.00,0.403836,0.123374,0,1.097798,auto,0,2012-06-18 09:31:54.393,2022-02


In [17]:
# preprocess
list_str_filename = [
    'preprocessing.py',
    'cls_model_preprocessing.pkl',
]
for str_filename in tqdm(list_str_filename):
    # download
    str_bucket_path = f'01_ad/02_model/{str_variant}/00_preprocessing/01_create_preprocessor/{str_filename}'
    str_local_path = f'./{str_filename}'
    download_from_s3(
        str_local_path=str_local_path, 
        str_bucket_path=str_bucket_path, 
        str_project=str_project,
    )
# import
cls_model_preprocessing = pickle.load(open(str_local_path, 'rb'))
# rm
os.remove(str_local_path)

# preprocess
df = cls_model_preprocessing.transform(df)

# rm
os.remove('./preprocessing.py')

# show
df

100%|██████████| 2/2 [00:00<00:00,  6.45it/s]


NaN Replacer: 0.033576 sec.


100%|██████████| 3/3 [00:00<00:00, 761.35it/s]


Unable to convert vehiclemodel__app to string, not found in data
Set strings: 0.0059203 sec.
Boolean Replacer: 0.089253 sec.


100%|██████████| 1987/1987 [00:00<00:00, 5785.50it/s]


...

100%|██████████| 1992/1992 [00:00<00:00, 3086.72it/s]


Replace inf and -inf with NaN: 0.99441 sec.
Imputer: 0.063079 sec.
Map term: 0.0039081 sec.
Map PTI: 0.0032827 sec.


100%|██████████| 9/9 [00:00<00:00, 2247.22it/s]

Round values: 0.0069513 sec.
Preprocessing Model: 2.7737 sec.


,uniqueid__app_x,strcity__app,strname__app,strzipcode__app,bitapproved__app,bitsystemdecline__app,bitfunded__app,applicationmonth__app,applicationquarter__app,bigdealerid__app,...,dealerstampcreation__app,year_month,year,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-payment_to_income,ENG-loan_to_value,ENG-vehicle_age,ENG-dealership_age
0,5838462__7328771__20211124,burlington,kentucky,41005,1.0,nan,1.0,11.0,4.0,2836.0,...,2013-05-08 08:47:59.997,2021-11,2021,1.080027,11,4,0.12,1.023810,3.0,8.550685
1,5874043__7369806__20211220,dallas,texas,75219,1.0,nan,1.0,12.0,4.0,1469.0,...,2010-07-07 16:51:47.837,2021-12,2021,1.080027,12,4,0.12,1.219512,3.0,11.460274
2,5827507__7315370__20211113,saintlouis,missouri,63112,1.0,nan,1.0,11.0,4.0,5473.0,...,2018-05-07 13:27:18.220,2021-11,2021,1.080027,11,4,0.09,1.166667,7.0,3.520548
3,5855839__7349888__20211211,collinsville,illinois,62234,1.0,nan,1.0,12.0,4.0,5235.0,...,2017-09-18 16:02:07.110,2021-12,2021,1.080027,12,4,0.09,1.269231,4.0,4.230137
4,5874497__7370346__20211220,chicago,illinois,60652,1.0,nan,1.0,12.0,4.0,4535.0,...,2016-03-15 15:36:18.820,2021-12,2021,1.080027,12,4,0.06,1.084746,3.0,5.767123
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9417,0__0__20220222,chandler,arizona,85225,0.0,0.0,0.0,2.0,1.0,0.0,...,2003-07-08 14:32:29.097,2022-02,2022,1.000000,2,1,0.12,1.270270,5.0,18.641096
9525,0__0__20220221,indianapolis,indiana,46237,0.0,0.0,0.0,2.0,1.0,0.0,...,2013-07-18 09:37:34.627,2022-02,2022,1.000000,2,1,0.06,1.000000,2.0,8.602740
9702,0__0__20220222,louisville,kentucky,40272,0.0,0.0,0.0,2.0,1.0,0.0,...,2012-11-21 13:38:42.487,2022-02,2022,1.000000,2,1,0.09,1.020408,3.0,9.260274
10670,0__0__20220214,cypress,louisiana,71457,0.0,0.0,0.0,2.0,1.0,0.0,...,2012-06-18 09:31:54.393,2022-02,2022,1.000000,2,1,0.12,1.219512,8.0,9.665753


In [18]:
# get gen 12 PD model
str_filename = 'final_model.pkl'
str_bucket_path = f'02_pricing_pd/02_model/{str_variant}/03_final_model/{str_filename}'
str_local_path = f'{str_dirname_output}/{str_filename}'
download_from_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)
cls_model_inference = pickle.load(open(str_local_path, 'rb'))['model_inference']
os.remove(str_local_path)
# predict
list_cols_model = list(cls_model_inference.feature_names_)
df['yhat_pd'] = cls_model_inference.predict_proba(df[list_cols_model])[:, 1]
# show
df

,uniqueid__app_x,strcity__app,strname__app,strzipcode__app,bitapproved__app,bitsystemdecline__app,bitfunded__app,applicationmonth__app,applicationquarter__app,bigdealerid__app,...,year_month,year,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-payment_to_income,ENG-loan_to_value,ENG-vehicle_age,ENG-dealership_age,yhat_pd
0,5838462__7328771__20211124,burlington,kentucky,41005,1.0,nan,1.0,11.0,4.0,2836.0,...,2021-11,2021,1.080027,11,4,0.12,1.023810,3.0,8.550685,0.288796
1,5874043__7369806__20211220,dallas,texas,75219,1.0,nan,1.0,12.0,4.0,1469.0,...,2021-12,2021,1.080027,12,4,0.12,1.219512,3.0,11.460274,0.520349
2,5827507__7315370__20211113,saintlouis,missouri,63112,1.0,nan,1.0,11.0,4.0,5473.0,...,2021-11,2021,1.080027,11,4,0.09,1.166667,7.0,3.520548,0.425721
3,5855839__7349888__20211211,collinsville,illinois,62234,1.0,nan,1.0,12.0,4.0,5235.0,...,2021-12,2021,1.080027,12,4,0.09,1.269231,4.0,4.230137,0.541364
4,5874497__7370346__20211220,chicago,illinois,60652,1.0,nan,1.0,12.0,4.0,4535.0,...,2021-12,2021,1.080027,12,4,0.06,1.084746,3.0,5.767123,0.118752
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9417,0__0__20220222,chandler,arizona,85225,0.0,0.0,0.0,2.0,1.0,0.0,...,2022-02,2022,1.000000,2,1,0.12,1.270270,5.0,18.641096,0.294556
9525,0__0__20220221,indianapolis,indiana,46237,0.0,0.0,0.0,2.0,1.0,0.0,...,2022-02,2022,1.000000,2,1,0.06,1.000000,2.0,8.602740,0.091096
9702,0__0__20220222,louisville,kentucky,40272,0.0,0.0,0.0,2.0,1.0,0.0,...,2022-02,2022,1.000000,2,1,0.09,1.020408,3.0,9.260274,0.328182
10670,0__0__20220214,cypress,louisiana,71457,0.0,0.0,0.0,2.0,1.0,0.0,...,2022-02,2022,1.000000,2,1,0.12,1.219512,8.0,9.665753,0.435115


In [19]:
# get gen 12 LGD model
str_filename = 'final_model.pkl'
str_bucket_path = f'03_pricing_lgd/02_model/{str_variant}/03_final_model/{str_filename}'
str_local_path = f'{str_dirname_output}/{str_filename}'
download_from_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)
cls_model_inference = pickle.load(open(str_local_path, 'rb'))['model_inference']
os.remove(str_local_path)
# predict
list_cols_model = list(cls_model_inference.feature_names_)
df['yhat_lgd'] = cls_model_inference.predict(df[list_cols_model])
# show
df

,uniqueid__app_x,strcity__app,strname__app,strzipcode__app,bitapproved__app,bitsystemdecline__app,bitfunded__app,applicationmonth__app,applicationquarter__app,bigdealerid__app,...,year,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-payment_to_income,ENG-loan_to_value,ENG-vehicle_age,ENG-dealership_age,yhat_pd,yhat_lgd
0,5838462__7328771__20211124,burlington,kentucky,41005,1.0,nan,1.0,11.0,4.0,2836.0,...,2021,1.080027,11,4,0.12,1.023810,3.0,8.550685,0.288796,0.395339
1,5874043__7369806__20211220,dallas,texas,75219,1.0,nan,1.0,12.0,4.0,1469.0,...,2021,1.080027,12,4,0.12,1.219512,3.0,11.460274,0.520349,0.489829
2,5827507__7315370__20211113,saintlouis,missouri,63112,1.0,nan,1.0,11.0,4.0,5473.0,...,2021,1.080027,11,4,0.09,1.166667,7.0,3.520548,0.425721,0.395133
3,5855839__7349888__20211211,collinsville,illinois,62234,1.0,nan,1.0,12.0,4.0,5235.0,...,2021,1.080027,12,4,0.09,1.269231,4.0,4.230137,0.541364,0.509507
4,5874497__7370346__20211220,chicago,illinois,60652,1.0,nan,1.0,12.0,4.0,4535.0,...,2021,1.080027,12,4,0.06,1.084746,3.0,5.767123,0.118752,0.420160
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9417,0__0__20220222,chandler,arizona,85225,0.0,0.0,0.0,2.0,1.0,0.0,...,2022,1.000000,2,1,0.12,1.270270,5.0,18.641096,0.294556,0.436393
9525,0__0__20220221,indianapolis,indiana,46237,0.0,0.0,0.0,2.0,1.0,0.0,...,2022,1.000000,2,1,0.06,1.000000,2.0,8.602740,0.091096,0.252261
9702,0__0__20220222,louisville,kentucky,40272,0.0,0.0,0.0,2.0,1.0,0.0,...,2022,1.000000,2,1,0.09,1.020408,3.0,9.260274,0.328182,0.360731
10670,0__0__20220214,cypress,louisiana,71457,0.0,0.0,0.0,2.0,1.0,0.0,...,2022,1.000000,2,1,0.12,1.219512,8.0,9.665753,0.435115,0.414417


In [20]:
# get ecnl
df_grouped = df.groupby(by='bigaccountid__app', as_index=False).agg({
    'yhat_pd': 'mean',
    'yhat_lgd': 'mean',
})
df_grouped['ecnl'] = df_grouped['yhat_pd'] * df_grouped['yhat_lgd']

# Mike's adjustment
df_grouped['ecnl'] = (1.95553 * df_grouped['ecnl']) - 0.03281

# get tier
df_grouped['tier'] = df_grouped['ecnl'].apply(get_tier)
# subset
list_cols = [
    'bigaccountid__app',
    'tier',
]
df_grouped = df_grouped[list_cols].copy()
# show
df_grouped

,bigaccountid__app,tier
0,5798398.0,Decline
1,5800475.0,Decline
2,5808212.0,Decline
3,5809809.0,Decline
4,5811416.0,Decline
...,...,...
3888,5951271.0,Decline
3889,5951279.0,C
3890,5951280.0,D
3891,5951307.0,Decline


In [21]:
# join back to df_tmp
df = pd.merge(
    left=df,
    right=df_grouped,
    on='bigaccountid__app',
    how='inner',
)
df

,uniqueid__app_x,strcity__app,strname__app,strzipcode__app,bitapproved__app,bitsystemdecline__app,bitfunded__app,applicationmonth__app,applicationquarter__app,bigdealerid__app,...,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-payment_to_income,ENG-loan_to_value,ENG-vehicle_age,ENG-dealership_age,yhat_pd,yhat_lgd,tier
0,5838462__7328771__20211124,burlington,kentucky,41005,1.0,nan,1.0,11.0,4.0,2836.0,...,1.080027,11,4,0.12,1.023810,3.0,8.550685,0.288796,0.395339,D
1,5874043__7369806__20211220,dallas,texas,75219,1.0,nan,1.0,12.0,4.0,1469.0,...,1.080027,12,4,0.12,1.219512,3.0,11.460274,0.520349,0.489829,Decline
2,5827507__7315370__20211113,saintlouis,missouri,63112,1.0,nan,1.0,11.0,4.0,5473.0,...,1.080027,11,4,0.09,1.166667,7.0,3.520548,0.425721,0.395133,Decline
3,5855839__7349888__20211211,collinsville,illinois,62234,1.0,nan,1.0,12.0,4.0,5235.0,...,1.080027,12,4,0.09,1.269231,4.0,4.230137,0.541364,0.509507,Decline
4,5874497__7370346__20211220,chicago,illinois,60652,1.0,nan,1.0,12.0,4.0,4535.0,...,1.080027,12,4,0.06,1.084746,3.0,5.767123,0.118752,0.420160,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4891,0__0__20220222,chandler,arizona,85225,0.0,0.0,0.0,2.0,1.0,0.0,...,1.000000,2,1,0.12,1.270270,5.0,18.641096,0.294556,0.436393,Decline
4892,0__0__20220221,indianapolis,indiana,46237,0.0,0.0,0.0,2.0,1.0,0.0,...,1.000000,2,1,0.06,1.000000,2.0,8.602740,0.091096,0.252261,A1
4893,0__0__20220222,louisville,kentucky,40272,0.0,0.0,0.0,2.0,1.0,0.0,...,1.000000,2,1,0.09,1.020408,3.0,9.260274,0.328182,0.360731,D
4894,0__0__20220214,cypress,louisiana,71457,0.0,0.0,0.0,2.0,1.0,0.0,...,1.000000,2,1,0.12,1.219512,8.0,9.665753,0.435115,0.414417,Decline


In [22]:
# subset to tier
df = df[df['tier'] == 'A1'].copy()
# subse to bk type
df = df[df['intopenbktype__app'] == 'nan'].copy()
# show
df

,uniqueid__app_x,strcity__app,strname__app,strzipcode__app,bitapproved__app,bitsystemdecline__app,bitfunded__app,applicationmonth__app,applicationquarter__app,bigdealerid__app,...,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-payment_to_income,ENG-loan_to_value,ENG-vehicle_age,ENG-dealership_age,yhat_pd,yhat_lgd,tier
84,5877868__7374436__20211223,orem,utah,84058,1.0,nan,1.0,12.0,4.0,1589.0,...,1.080027,12,4,0.09,1.093750,4.0,11.104110,0.105527,0.288649,A1
90,5881719__7379126__20211229,lehi,utah,84043,1.0,nan,1.0,12.0,4.0,5454.0,...,1.080027,12,4,0.06,1.305556,8.0,3.684932,0.110916,0.265454,A1
157,5876732__7373048__20211222,saltlakecity,utah,84118,1.0,nan,1.0,12.0,4.0,78.0,...,1.080027,12,4,0.15,1.444444,4.0,17.208219,0.086332,0.250205,A1
286,5880825__7378038__20211228,magna,utah,84044,1.0,nan,1.0,12.0,4.0,5466.0,...,1.080027,12,4,0.12,1.181818,7.0,3.660274,0.098165,0.318542,A1
362,5885259__7383502__20220103,southbend,indiana,46614,1.0,nan,1.0,1.0,1.0,4747.0,...,1.000000,1,1,0.12,1.064516,5.0,5.312329,0.125656,0.292834,A1
508,5889069__7388160__20220107,chicago,illinois,60608,1.0,nan,1.0,1.0,1.0,5406.0,...,1.000000,1,1,0.06,1.122807,3.0,3.852055,0.097933,0.234348,A1
524,5881822__7379250__20211229,grantsville,utah,84029,1.0,nan,1.0,12.0,4.0,2506.0,...,1.080027,12,4,0.15,1.457627,6.0,9.142466,0.146315,0.247601,A1
530,5881822__7379251__20211229,grantsville,utah,84029,1.0,nan,1.0,12.0,4.0,2506.0,...,1.080027,12,4,0.15,1.457627,6.0,9.142466,0.181903,0.165772,A1
610,5887790__7386599__20220105,saltlakecty,utah,84101,1.0,nan,1.0,1.0,1.0,5454.0,...,1.000000,1,1,0.03,1.120000,9.0,3.704110,0.099367,0.279621,A1
782,5879712__7376668__20211227,saltlakecty,utah,84109,1.0,nan,1.0,12.0,4.0,71.0,...,1.080027,12,4,0.12,1.250000,7.0,18.484932,0.124227,0.256199,A1


### Get actual targets

In [23]:
# get actual targets
str_filename = 'df_early_targets.csv'
str_uri = f's3://{str_project}/09_early_indicators/05_get_target_from_db/{str_filename}'
df_tmp = pd.read_csv(str_uri)

# join
df = pd.merge(
    left=df,
    right=df_tmp,
    on='bigAccountId',
    how='inner',
)

dict_prod_target_mean = {}
for key, val in tqdm(dict_model_column.items()):
    dict_prod_target_mean[key] = df[val].mean()

# show
dict_prod_target_mean

100%|██████████| 115/115 [00:00<00:00, 18580.31it/s]


{'DQ1_1': 0.0,
 'DQ1_2': 0.3877551020408163,
 'DQ1_3': 0.5306122448979592,
 'DQ1_4': 0.6530612244897959,
 'DQ1_5': 0.673469387755102,
 'DQ1_6': 0.673469387755102,
 'DQ1_7': 0.7346938775510204,
 'DQ1_8': 0.7551020408163265,
 'DQ1_9': 0.7551020408163265,
 'DQ1_10': 0.7551020408163265,
 'DQ1_11': 0.7551020408163265,
 'DQ1_12': 0.7551020408163265,
 'DQ1_13': 0.7551020408163265,
 'DQ1_14': 0.7551020408163265,
 'DQ1_15': 0.7551020408163265,
 'DQ1_16': 0.7755102040816326,
 'DQ1_17': 0.7755102040816326,
 'DQ1_18': 0.7755102040816326,
 'DQ1_19': 0.7959183673469388,
 'DQ1_20': 0.7959183673469388,
 'DQ1_21': 0.7959183673469388,
 'DQ1_22': 0.7959183673469388,
 'DQ1_23': 0.8367346938775511,
 'DQ1_24': 0.8367346938775511,
 'DQ15_1': 0.0,
 'DQ15_2': 0.061224489795918366,
 'DQ15_3': 0.10204081632653061,
 'DQ15_4': 0.14285714285714285,
 'DQ15_5': 0.1836734693877551,
 'DQ15_6': 0.22448979591836735,
 'DQ15_7': 0.2653061224489796,
 'DQ15_8': 0.2857142857142857,
 'DQ15_9': 0.2857142857142857,
 'DQ15_10': 0

### Summary

In [24]:
list_dict_row = []
for key, val in tqdm(dict_model_column.items()):
    # get mean training actual
    flt_mean_train_actual = dict_train_target_mean[key]
    # get mean of production actuals
    flt_mean_prod_actual = dict_prod_target_mean[key]
    # dict_row
    dict_row = {
        'indicator': key,
        'mean_train_actual': flt_mean_train_actual,
        'mean_prod_actual': flt_mean_prod_actual,
    }
    list_dict_row.append(dict_row)
    
# make df 
df_summary = pd.DataFrame(list_dict_row)

# dates
df_summary['min_date'] = list_str_year_month[0]
df_summary['max_date'] = list_str_year_month[-1]

# get days delinquent
df_summary['days_delinquent'] = df_summary['indicator'].apply(
    lambda x: int(x.split('_')[0][2:]),
)

# get days total
df_summary['days_total'] = df_summary['indicator'].apply(
    lambda x: int(x.split('_')[1]) * 30, # 30 days per month
)

# show
df_summary

100%|██████████| 115/115 [00:00<00:00, 583246.63it/s]


,indicator,mean_train_actual,mean_prod_actual,min_date,max_date,days_delinquent,days_total
0,DQ1_1,0.095332,0.000000,2021-10,2022-02,1,30
1,DQ1_2,0.191321,0.387755,2021-10,2022-02,1,60
2,DQ1_3,0.270874,0.530612,2021-10,2022-02,1,90
3,DQ1_4,0.351085,0.653061,2021-10,2022-02,1,120
4,DQ1_5,0.406969,0.673469,2021-10,2022-02,1,150
...,...,...,...,...,...,...,...
110,DQ90_20,0.019066,0.081633,2021-10,2022-02,90,600
111,DQ90_21,0.022354,0.081633,2021-10,2022-02,90,630
112,DQ90_22,0.023669,0.102041,2021-10,2022-02,90,660
113,DQ90_23,0.024984,0.102041,2021-10,2022-02,90,690


### Save

In [25]:
str_filename = 'df_summary.csv'
str_local_path = f'{str_dirname_output}/{str_variant}/{str_filename}'
df_summary.to_csv(str_local_path, index=False)